# Practical P14: JSON Parsing: Nested Structures
**Learning Outcome**: Navigate any JSON API response and extract needed fields safely.

## Part 1: Safely Navigating Dictionary Paths using `.get()`
If an API returns a schema without certain fields, referencing `dict['key']` triggers a `KeyError` and crashes your application.
Using `dict.get('key', default_value)` returns a default value instead of crashing.


In [ ]:
response_data = {
    'status': 'success',
    'results': {
        'model_version': '1.0.0'
    }
}

# Dangerous lookup:
try:
    print(response_data['results']['safety_score'])
except KeyError as e:
    print('KeyError triggered on:', e)

# Safe lookup:
results = response_data.get('results', {})
safety_score = results.get('safety_score', 'N/A')
print('Safe extraction of missing key safety_score:', safety_score)


## Part 2: Extracting Nested Records in a Loop
Let's load the sample response JSON file we created in our dataset directory.


In [ ]:
import json
with open('data/mock_response_samples.json', 'r') as f:
    samples = json.load(f)

print('Loaded sample configurations:', list(samples.keys()))


## Hands-On Exercise
**Task**: Load `openai_success` and `anthropic_success` mock JSONs from `data/mock_response_samples.json`.
Write an extractor function `get_model_and_tokens(payload, provider)` that parses details based on provider name and returns a unified dict format:
`{'provider': X, 'model': Y, 'prompt_tokens': A, 'completion_tokens': B, 'response_text': C}`

Note:
- OpenAI formats input/output as `usage.prompt_tokens` and `usage.completion_tokens`, content at `choices[0].message.content`.
- Anthropic formats input/output as `usage.input_tokens` and `usage.output_tokens`, content at `content[0].text`.


In [ ]:
# TODO: Build get_model_and_tokens
def get_model_and_tokens(payload, provider):
    if provider == 'openai':
        usage = payload.get('usage', {})
        choices = payload.get('choices', [{}])
        msg = choices[0].get('message', {})
        return {
            'provider': 'openai',
            'model': payload.get('model'),
            'prompt_tokens': usage.get('prompt_tokens', 0),
            'completion_tokens': usage.get('completion_tokens', 0),
            'response_text': msg.get('content', '')
        }
    elif provider == 'anthropic':
        usage = payload.get('usage', {})
        content_list = payload.get('content', [{}])
        return {
            'provider': 'anthropic',
            'model': payload.get('model'),
            'prompt_tokens': usage.get('input_tokens', 0),
            'completion_tokens': usage.get('output_tokens', 0),
            'response_text': content_list[0].get('text', '')
        }
    return None

print('OpenAI Extract:\n', get_model_and_tokens(samples['openai_success'], 'openai'))
print('\nAnthropic Extract:\n', get_model_and_tokens(samples['anthropic_success'], 'anthropic'))
